## Section 1: Theoretical Questions

In this section we first investigate core questions regarding auto-encoder networks, for example, how to achieve a vector representation for the deep features. Afterward, we discuss some characteristics of approaches to open-set recognition.

### Task 1.1: PCA vs. Two-Layer Autoencoder

Principal Component Analysis (PCA) and two-layer AutoEncoders (AE) are two different ways for projecting a vector input vector $\vec x \in \mathbb R^D$ into a vector representation $\vec \varphi \in \mathbb R^K$ with $K \ll D$.
In both cases, a projection matrix $\mathbf W$ is learned to perform this task.
For PCA, this projection is applied as $\vec\varphi = \mathbf W(\vec x - \tilde x)$ where $\tilde x$ is the mean of the samples over the training set.
A two-layer autoencoder is composed of an encoder with one fully-connected layer $\mathbf W^{(1)} \in \mathbb R^{K\times(D+1)}$, and a decoder containing one fully-connected layer $\mathbf W^{(2)} \in \mathbb R^{(K+1)\times D}$, each including a bias neuron.

**Question**: What is the difference between PCA and the two-layer autoencoder with respect to the embedding matrix $\mathbf W$, and the learning procedure? Is there any theoretical difference how the bias is handled between the two methods? 

**Answers** 
While seemingly similar, PCA and two-layer autoencoders have key differences:

- PCA learns orthogonal vectors (eigenvectors), while a two-layer autoencoder can -- but is not guaranteed to -- learn orthogonal representations.
  In particular, a linear two-layer autoencoder can theoretically converge to the same subspace as PCA, but may find different basis vectors.
- PCA minimizes reconstruction error under orthogonality constraints, while autoencoders minimize reconstruction error without such constraints.
- The principal components from PCA are the directions of maximum variance (sorted in descending order), while autoencoders can learn representations that capture different aspects of the data. This makes autoencoders more flexible in terms of the types of features they can learn, but it loses the interpretability of PCA
- PCA has a closed-form solution, while autoencoders require iterative optimization
- The bias in PCA is handled in the input space by subtracting the mean $\tilde x$, whereas the two-layer AE handles the bias in *feature space* by adding $\mathbf W^{(1)}_{:,0}$ to the embedding. For PCA, the bias $\tilde x$ is directly computed from the data, in the AE it is learned through back-propagation. The effect is similar, but not identical.

### Task 1.2: Options for One-Dimensional Feature Vectors in Latent Space

In a convolutional autoencoder, the encoder typically employs a series of convolutional layers, which provides feature maps $\mathcal A\in\mathbb R^{Q\times K\times M}$. 
The embedding vector, however, is required to be a vector with a desired number of elements.

**Question**: What are the different approaches to obtaining a one-dimensional feature vector in the latent space of autoencoders?

**Answers**:
Several methods exist to reduce dimensionality to a one-dimensional feature vector:

- **Linear layer**: Using pooling/flattening and a fully connected layer to project to a single dimension
  - Pros: Simple, learnable weights
  - Cons: Adds parameters, may lose spatial information

- **Flattening**: Flattening multi-dimensional tensor to 1D vector
  - Pros: Preserves all information, easily invertible
  - Cons: Results in high-dimensional vectors of not precisely specifiable number of elements, loses spatial relationship

- **Global Average Pooling (GAP)**: Averaging across spatial dimensions
  - Pros: Parameter-free, translation invariant
  - Cons: Harder to invert for reconstruction, loses spatial information
  - Note: Generally not reversible for autoencoders that need to reconstruct input

- **Architecture design**: Designing the encoder with specific kernel sizes, padding, and stride to reduce the entire input to a single dimension with multiple channels
  - Pros: Naturally reduces dimensions through convolution operations
  - Cons: Requires careful architecture design, less flexible, relies on specific input sizes
  - Implementation: Create a network that gradually reduces spatial dimensions to $A\in\mathbb R^{Q\times1\times1}$ ($Q$ channels of $1\times1$ images)

### Task 1.3: Two-Step Approach for OSR - Pros and Cons

In open-set recognition, two-stage approaches first perform a binary classification of a sample to be *known* or *unknown*, and then classify all *known* samples as their respective class.

**Question**: What are the advantages and disadvantages of employing such a two-step approach for Open-Set Recognition?

**Answers**: 

**Pros**:
- Modularity allows optimization of each component separately
- Can leverage existing closed-set classifiers without modifying them
- Easier to understand and debug since steps are separated
- Can handle unknown classes without retraining the entire system by adding a OOD detector
- May be more computationally efficient during inference

**Cons**:
- Potential error propagation between steps
- May fail to leverage interdependencies between classification and OOD detection
- Requires tuning multiple thresholds or hyperparameters
- Typically requires more parameters than end-to-end approaches
- Two separate optimization objectives might not align optimally

### Task 1.4: Entropic Open-Set (EOS) Loss Bias

The Entropic Open-Set (EOS) loss aka. the Softmax Adaptation requires having no bias in the final fully-connected layer that transforms the deep features $\vec\varphi$ to the logits: $\vec z = W^{(L)} \vec\varphi$.

**Question**: Why is it theoretically impossible to introduce a bias for the Entropic Open-Set (EOS) loss?

**Answer**:
The EOS loss naturally encourages the distribution of unknown classes to become a uniform distribution over the output space.
As such, the unknown samples are drawn to the origin of the deep feature space, while the known classes are radially distributed around the origin.
Introducing a bias term would shift the samples away from the origin, which would counteract the EOS loss's goal of clustering unknowns at the origin.
In practice, if bias weights are learned to be identical for all classes, this results in logits not being zero, but still identical when deep feature vectors are vanishing.

### Task 1.5: Selection of Negative Samples

EOS loss requires *negative* samples for training.

**Question**: Given a specific set of known classes, how should negative samples be selected? Can random noise be used as negatives?

**Answer**: 
The selection of negative samples is critical for OSR performance:

- Random noise is generally a poor choice as it's too easily distinguished from real data
- Better approaches include:
  - Using samples from related but distinct datasets
  - Synthetic samples generated via data augmentation, noise *addition*, or generative models
  - Samples from a diverse "background" class
  - Out-of-distribution samples that maintain the same general semantics
- The negative samples do not need to be diverse enough to cover the potential unknown space, for certain losses it suffices to learn decision boundaries that are close to the known classes, which can be achieved with negatives that are similar to known classes

### Task 1.6: Training with Negative Samples for Different OOD Detection Approaches

As mentioned above, we will make use of two methods for performing Out-of-Distribution (OOD) detection.
One is based on thresholding reconstruction error values, and one is based on distances in deep feature space.
More details on these two methods can be found at later sections in this notebook.

Theoretically, we could add the training samples of our unknown datasets (EMNIST letters, FashionMNIST) to our original training data (MNIST), and use all of these to train our autoencoder network.

**Question**: What do you expect: Is it advisable to train with additional samples when using autoencoders for OOD detection? Is there a difference between the two methods for OOD, i.e., thresholding reconstruction error or thresholding distances to class centroids in the deep feature space?

**Answer**: ...
The benefit of additional samples depends on the OOD approach:

- **When using reconstruction error for OOD detection**:
  - Not advisable to train with samples of unknown classes
  - The autoencoder would learn to reconstruct such samples
  - In worst case, it might learn to reconstruct *other unknowns* as well. In this case we don't want the network to generalize to unknowns.
  - Adding such training samples actually hurts performance
  - Requires labeled data to ensure that the training data is representative of the known classes only. This eliminates the key benefit of unsupervised learning not requiring labeled data

- **When using the deep feature space for OOD detection**:
  - Beneficial to train with additional samples
  - Helps learn more relevant features that aid unknown detection
  - Creates more discriminative latent representations
  - Encourages known classes to cluster more tightly while leaving space for unknowns